# VLM + MCP = The "Brain" Concept

**HFA AI Training — Module 6: Vision Language Models**

This is the most important concept in this module.

**THE BIG IDEA:**
- A VLM (Vision Language Model) can SEE and UNDERSTAND images
- An MCP (Model Context Protocol) gives AI TOOLS to take actions
- Combined, you get an agent that can PERCEIVE and ACT — a "Brain"

**THE PIPELINE:**
1. **SEE** — The VLM looks at an image (photo, document, video frame)
2. **THINK** — The VLM reasons about what it sees, extracts information
3. **ACT** — MCP tools take action based on that information

**REAL-WORLD ANALOGY:**

Think of how your brain works:
- Your EYES see a stop sign (perception)
- Your BRAIN recognizes it as a stop sign and knows the rule (reasoning)
- Your FOOT presses the brake (action)

VLM + MCP works the same way:
- VLM sees the image (perception)
- VLM understands what it's looking at (reasoning)
- MCP tool takes the appropriate action (action)

This notebook demonstrates the full see-think-act pipeline using:
- Gemini as the VLM (for seeing and reasoning)
- Simulated MCP tools (for taking actions)

In a real deployment, the MCP tools would connect to real systems
(databases, file systems, email, CRMs, etc.).

## Install Dependencies

In [ ]:
!pip install google-generativeai httpx

## Imports and Configuration

In [ ]:
import os
import json
import httpx
from datetime import datetime

import google.generativeai as genai

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise EnvironmentError(
        "GOOGLE_API_KEY not set. Run: export GOOGLE_API_KEY='your-key'\n"
        "Get a free key at https://aistudio.google.com/apikey"
    )

genai.configure(api_key=GOOGLE_API_KEY)
MODEL_NAME = "gemini-2.0-flash"

## Part 1: The MCP Tools (Simulated)

In a real system, these would be MCP server tools that connect to actual
services. Here we simulate them to show the concept.

The key insight: these tools don't need to "see" — they just take action
based on structured data that the VLM extracted from images.

In [ ]:
class SimulatedMCPTools:
    """
    Simulated MCP tools that represent real-world actions.

    In production, each of these would be an MCP tool served by an MCP server
    that connects to actual systems (database, file storage, CRM, email, etc.).

    The key insight: these tools don't need to "see" — they just take action
    based on structured data that the VLM extracted from images.
    """

    @staticmethod
    def file_document(document_type: str, extracted_data: dict, source: str) -> dict:
        """
        MCP Tool: File a document in the appropriate location.

        In reality, this would:
          - Save to a document management system
          - Upload to Google Drive / Dropbox
          - Store in a database with metadata
        """
        print(f"\n  [MCP TOOL] Filing document...")
        print(f"    Type:   {document_type}")
        print(f"    Source: {source}")
        print(f"    Data:   {json.dumps(extracted_data, indent=6)}")

        # Simulate filing the document.
        result = {
            "status": "filed",
            "document_id": f"DOC-{datetime.now().strftime('%Y%m%d%H%M%S')}",
            "filed_as": document_type,
            "location": f"/documents/{document_type.lower().replace(' ', '_')}/",
            "timestamp": datetime.now().isoformat(),
        }
        print(f"    Result: Document filed as {result['document_id']}")
        return result

    @staticmethod
    def create_listing(property_data: dict) -> dict:
        """
        MCP Tool: Create a property listing from extracted data.

        In reality, this would:
          - Create a listing in the MLS system
          - Update the CRM
          - Generate marketing materials
        """
        print(f"\n  [MCP TOOL] Creating property listing...")
        print(f"    Property: {property_data.get('property_type', 'Unknown')}")
        print(f"    Style:    {property_data.get('style', 'Unknown')}")
        print(f"    Features: {property_data.get('features', [])}")

        result = {
            "status": "listing_created",
            "listing_id": f"MLS-{datetime.now().strftime('%Y%m%d%H%M%S')}",
            "property_data": property_data,
            "timestamp": datetime.now().isoformat(),
        }
        print(f"    Result: Listing created as {result['listing_id']}")
        return result

    @staticmethod
    def send_notification(recipient: str, subject: str, body: str) -> dict:
        """
        MCP Tool: Send a notification (email, SMS, Slack, etc.).

        In reality, this would connect to an email service, Twilio, or Slack.
        """
        print(f"\n  [MCP TOOL] Sending notification...")
        print(f"    To:      {recipient}")
        print(f"    Subject: {subject}")
        print(f"    Body:    {body[:100]}...")

        result = {
            "status": "sent",
            "recipient": recipient,
            "subject": subject,
            "timestamp": datetime.now().isoformat(),
        }
        print(f"    Result: Notification sent to {recipient}")
        return result

    @staticmethod
    def flag_for_review(item: str, reason: str, priority: str) -> dict:
        """
        MCP Tool: Flag something for human review.

        Sometimes the right action is to escalate to a human.
        """
        print(f"\n  [MCP TOOL] Flagging for review...")
        print(f"    Item:     {item}")
        print(f"    Reason:   {reason}")
        print(f"    Priority: {priority}")

        result = {
            "status": "flagged",
            "review_id": f"REV-{datetime.now().strftime('%Y%m%d%H%M%S')}",
            "priority": priority,
            "timestamp": datetime.now().isoformat(),
        }
        print(f"    Result: Flagged as {result['review_id']} ({priority} priority)")
        return result

## Part 2: The VLM "Brain" — See, Think, Act

This is the full "Brain" pipeline:
1. **SEE** — Sends an image to Gemini (the VLM)
2. **THINK** — Gemini analyzes and extracts structured data
3. **ACT** — Based on the analysis, calls the right MCP tools

In [ ]:
def vlm_brain_pipeline(image_url: str, context: str = "") -> dict:
    """
    The full "Brain" pipeline: SEE -> THINK -> ACT.

    This function:
      1. Sends an image to Gemini (the VLM) — SEE
      2. Gemini analyzes and extracts structured data — THINK
      3. Based on the analysis, calls the right MCP tools — ACT

    Args:
        image_url: URL of the image to process.
        context: Optional context about what this image is
                 (e.g., "new property listing photo", "expense receipt").

    Returns:
        Dictionary with the full pipeline results.
    """
    tools = SimulatedMCPTools()

    print("\n" + "=" * 70)
    print("  VLM + MCP BRAIN PIPELINE: SEE -> THINK -> ACT")
    print("=" * 70)

    # -----------------------------------------------------------------------
    # STEP 1: SEE — Send the image to the VLM
    # -----------------------------------------------------------------------
    print("\n--- STEP 1: SEE (VLM perceives the image) ---")
    print(f"  Sending image to Gemini for analysis...")
    print(f"  Image: {image_url}")
    if context:
        print(f"  Context: {context}")

    # Download the image from the URL.
    image_response = httpx.get(image_url)
    image_bytes = image_response.content

    analysis_prompt = f"""Analyze this image and provide a structured assessment.

    {f'Context: {context}' if context else ''}

    Respond with ONLY a JSON object containing:
    {{
        "image_type": "property_photo | document | receipt | floor_plan | other",
        "confidence": 0.0 to 1.0,
        "summary": "one-line description of what you see",
        "detailed_analysis": "thorough analysis of the image",
        "extracted_data": {{
            // all relevant structured data you can extract
        }},
        "recommended_actions": [
            {{
                "action": "file_document | create_listing | send_notification | flag_for_review",
                "reason": "why this action should be taken",
                "parameters": {{}}
            }}
        ],
        "concerns": ["any issues or things that need attention"]
    }}

    Return ONLY valid JSON."""

    model = genai.GenerativeModel(MODEL_NAME)

    response = model.generate_content(
        [
            {"mime_type": "image/jpeg", "data": image_bytes},
            analysis_prompt,
        ]
    )

    raw_text = response.text.strip()

    # Clean up markdown code blocks if present.
    if raw_text.startswith("```"):
        raw_text = raw_text.split("\n", 1)[1]
        raw_text = raw_text.rsplit("```", 1)[0]

    try:
        vlm_analysis = json.loads(raw_text)
    except json.JSONDecodeError:
        print(f"  Warning: Could not parse VLM response as JSON.")
        print(f"  Raw response: {raw_text[:200]}...")
        vlm_analysis = {
            "image_type": "other",
            "summary": raw_text[:100],
            "recommended_actions": [],
        }

    print(f"\n  VLM sees: {vlm_analysis.get('summary', 'N/A')}")
    print(f"  Image type: {vlm_analysis.get('image_type', 'N/A')}")
    print(f"  Confidence: {vlm_analysis.get('confidence', 'N/A')}")

    # -----------------------------------------------------------------------
    # STEP 2: THINK — Process and decide on actions
    # -----------------------------------------------------------------------
    print("\n--- STEP 2: THINK (Reasoning about what was seen) ---")

    detailed = vlm_analysis.get("detailed_analysis", "No detailed analysis.")
    print(f"  Analysis: {detailed[:200]}...")

    recommended_actions = vlm_analysis.get("recommended_actions", [])
    print(f"  Recommended actions: {len(recommended_actions)}")
    for i, action in enumerate(recommended_actions, 1):
        print(f"    {i}. {action.get('action', 'unknown')} — {action.get('reason', 'no reason')}")

    concerns = vlm_analysis.get("concerns", [])
    if concerns:
        print(f"  Concerns flagged:")
        for concern in concerns:
            print(f"    - {concern}")

    # -----------------------------------------------------------------------
    # STEP 3: ACT — Execute MCP tools based on VLM's analysis
    # -----------------------------------------------------------------------
    print("\n--- STEP 3: ACT (MCP tools execute actions) ---")

    action_results = []

    for action in recommended_actions:
        action_type = action.get("action", "")
        params = action.get("parameters", {})

        if action_type == "file_document":
            result = tools.file_document(
                document_type=params.get("document_type", vlm_analysis.get("image_type", "unknown")),
                extracted_data=vlm_analysis.get("extracted_data", {}),
                source=image_url,
            )
            action_results.append(result)

        elif action_type == "create_listing":
            result = tools.create_listing(
                property_data=vlm_analysis.get("extracted_data", {}),
            )
            action_results.append(result)

        elif action_type == "send_notification":
            result = tools.send_notification(
                recipient=params.get("recipient", "team@example.com"),
                subject=params.get("subject", f"VLM Alert: {vlm_analysis.get('summary', 'Image analyzed')}"),
                body=vlm_analysis.get("detailed_analysis", "See attached analysis."),
            )
            action_results.append(result)

        elif action_type == "flag_for_review":
            result = tools.flag_for_review(
                item=vlm_analysis.get("summary", "Unknown item"),
                reason=action.get("reason", "Flagged by VLM analysis"),
                priority=params.get("priority", "medium"),
            )
            action_results.append(result)

        else:
            print(f"\n  [SKIP] Unknown action type: {action_type}")

    # If the VLM didn't recommend specific actions, take default actions
    # based on the image type.
    if not recommended_actions:
        print("  No specific actions recommended. Taking default action...")
        image_type = vlm_analysis.get("image_type", "other")

        if image_type == "property_photo":
            result = tools.create_listing(vlm_analysis.get("extracted_data", {}))
            action_results.append(result)
        elif image_type in ("document", "receipt"):
            result = tools.file_document(
                image_type, vlm_analysis.get("extracted_data", {}), image_url
            )
            action_results.append(result)
        else:
            result = tools.file_document(
                "general", vlm_analysis.get("extracted_data", {}), image_url
            )
            action_results.append(result)

    # -----------------------------------------------------------------------
    # SUMMARY
    # -----------------------------------------------------------------------
    print("\n--- PIPELINE COMPLETE ---")
    print(f"  Image analyzed: {vlm_analysis.get('summary', 'N/A')}")
    print(f"  Actions taken:  {len(action_results)}")
    for result in action_results:
        print(f"    - {result.get('status', 'unknown')}: "
              f"{result.get('document_id', result.get('listing_id', result.get('review_id', 'N/A')))}")

    return {
        "vlm_analysis": vlm_analysis,
        "actions_taken": action_results,
        "pipeline_completed": datetime.now().isoformat(),
    }

## Part 3: Example Scenarios

### Scenario 1: New Property Listing Photo

A real estate agent takes a photo of a new listing.
The Brain sees the property, extracts details, and creates a listing.

In [ ]:
def scenario_property_photo():
    """
    Scenario: A real estate agent takes a photo of a new listing.
    The Brain sees the property, extracts details, and creates a listing.
    """
    print("\n" + "#" * 70)
    print("  SCENARIO: New Property Listing Photo")
    print("  Agent takes a photo -> Brain creates the listing automatically")
    print("#" * 70)

    property_photo_url = (
        "https://upload.wikimedia.org/wikipedia/commons/thumb/"
        "1/19/Weatherboard_house_in_Coorparoo%2C_Queensland.jpg/"
        "1280px-Weatherboard_house_in_Coorparoo%2C_Queensland.jpg"
    )

    result = vlm_brain_pipeline(
        image_url=property_photo_url,
        context="This is a photo of a new property listing. "
                "Analyze the property and recommend creating a listing.",
    )

    return result

### Scenario 2: Incoming Document Processing

An office receives a document (invoice, form, etc.).
The Brain reads the document, extracts data, and files it.

NOTE: In this example we use a property photo as a stand-in since we
don't have a public domain document image URL handy. In a real system,
you would point this at actual document images.

In [ ]:
def scenario_document_processing():
    """
    Scenario: An office receives a document (invoice, form, etc.).
    The Brain reads the document, extracts data, and files it.

    NOTE: In this example we use a property photo as a stand-in since we
    don't have a public domain document image URL handy. In a real system,
    you would point this at actual document images.
    """
    print("\n" + "#" * 70)
    print("  SCENARIO: Incoming Document Processing")
    print("  Document arrives -> Brain reads it, extracts data, files it")
    print("#" * 70)

    # Using a sample image — in practice, this would be a document photo.
    document_url = (
        "https://upload.wikimedia.org/wikipedia/commons/thumb/"
        "e/e1/Ascot_house_1.jpg/"
        "1280px-Ascot_house_1.jpg"
    )

    result = vlm_brain_pipeline(
        image_url=document_url,
        context="Process this image. Extract all relevant information "
                "and file it appropriately.",
    )

    return result

## Run the Demonstrations

Execute the VLM + MCP "Brain" pipeline demonstrations.

In [ ]:
print("=" * 70)
print("  VLM + MCP = THE 'BRAIN' CONCEPT")
print("  See -> Think -> Act")
print("=" * 70)
print()
print("  THE IDEA:")
print("    VLM (Vision) = the eyes — perceives images and documents")
print("    MCP (Tools)  = the hands — takes actions in the real world")
print("    Together     = a Brain that can see, reason, and act")
print()
print("  PIPELINE:")
print("    1. SEE   — VLM looks at the image")
print("    2. THINK — VLM analyzes and extracts structured data")
print("    3. ACT   — MCP tools take action based on the analysis")
print()

### Run Scenario 1: Property Photo

In [ ]:
scenario_property_photo()

### Run Scenario 2: Document Processing

In [ ]:
scenario_document_processing()

In [ ]:
print("\n" + "=" * 70)
print("  BRAIN PIPELINE DEMONSTRATIONS COMPLETE")
print()
print("  Key takeaway: The VLM doesn't just see — it understands.")
print("  And with MCP tools, understanding leads to ACTION.")
print("  This is the future of AI automation.")
print("=" * 70)